In [ ]:
!ls -la /content
!ls -la /content/drive
!rm -rf /content/drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)



In [6]:
import os, sys, time

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName("7006SCN_HMDA2023_Task2")
    .master("local[*]")
    .config("spark.driver.memory", "10g")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

DATA    = "/content/drive/MyDrive/7006SCN/data/raw"
OUTPUT  = "/content/drive/MyDrive/7006SCN/outputs"
MODELS  = "/content/drive/MyDrive/7006SCN/models"

for p in [OUTPUT, MODELS]:
    os.makedirs(p, exist_ok=True)

CSV_PATH = f"{DATA}/My_Data.csv"

In [ ]:
# ── Commit 1: Ingestion + Repartition + Partition Count (CW mandatory) ─────────

from pyspark import StorageLevel
import time

start = time.time()

print("="*50)
print("HMDA 2023 DATA INGESTION")
print("="*50)

# Load dataset
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CSV_PATH)
)

# Cache immediately after loading
df.persist(StorageLevel.MEMORY_AND_DISK)

# Materialise cache
rows = df.count()
cols = len(df.columns)

load_time = time.time() - start

print(f"Rows               : {rows:,}")
print(f"Columns            : {cols}")
print(f"Load Time          : {load_time:.2f} seconds")
print()

print(f"Default partitions : {df.rdd.getNumPartitions()}")

# Repartition for balanced distributed processing
df = df.repartition(200)

print(f"After repartition  : {df.rdd.getNumPartitions()}")
print()

print(f"Storage Level      : {df.storageLevel}")

print()

print("Spark Version      :", spark.version)
print("Spark UI           :", spark.sparkContext.uiWebUrl)

print()

print("<-- Screenshot this output for Task 2 and Task 4 reports")

In [8]:
print(f"Estimated rows/partition : {rows//df.rdd.getNumPartitions():,}")

Estimated rows/partition : 57,820
